# 🏆 Hackathon — Build the Best RAG Pipeline You Can

You've built a working RAG pipeline and measured it. Now it's your turn to **beat your own benchmark**.

This notebook gives you a minimal scaffold: shared infrastructure (database, embedding model, test set, metrics) is pre-wired so you can focus entirely on the parts that matter. Your goal is to build a `run_rag(question)` function that returns `{question, contexts, answer}` and scores as high as possible across all six metrics — ideally in as little wall-clock time as possible.

```
alice_test_set.csv
       │
       ▼
 YOUR RAG PIPELINE          ← this is the only thing you need to build
       │
       ▼
 { question, contexts, answer, reference }
       │
       ▼
  Offline Metrics  (same as Notebook 7 — nothing changes here)
  ┌────────────┬───────────────────────┐
  │ Retrieval  │ Generation            │
  │ • Hit Rate │ • ROUGE-L             │
  │ • MRR      │ • BERTScore           │
  │ • Mean /   │                       │
  │   Max Tok  │                       │
  └────────────┴───────────────────────┘
```

---

### 🧭 Ideas to explore

You don't have to try all of these — pick whichever angle interests you most.

| Dimension | What to try |
|---|---|
| **Chunking** | Smaller or larger chunks? Sentence-level splitting? Overlapping windows? Semantic chunking? |
| **Embedding** | A different model (e.g. `BAAI/bge-large-en-v1.5`, `intfloat/e5-large-v2`)? Only dense? All three? |
| **Retrieval** | Different `top_k`? Only the best single retriever instead of ensemble? |
| **Fusion / RRF** | Tune the RRF constant (currently `k=60`)? Weight the three retrievers differently? |
| **Re-ranking** | Cross-encoder re-ranker (Notebook 6)? MMR for diversity? |
| **LLM / prompt** | A different Groq model? A tighter system prompt that forces concise answers? |
| **Speed** | Connection pooling? Async retrieval? Cache embeddings? |

> 💡 **Tip**: change one thing at a time and re-run the benchmark after each change so you can tell what actually helped.

---

## 0. Setup & Imports

Same boilerplate as the other notebooks — run this first.

In [1]:
import os
import csv
import json
import random
import time
import numpy as np
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
from dotenv import load_dotenv
from munch import Munch

# Offline metrics (identical to Notebook 7)
from rouge_score import rouge_scorer
from bert_score import score as bert_score

# Embedding model
from FlagEmbedding import BGEM3FlagModel

# LangChain
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [2]:
# Load config & secrets
config_path = (Path.cwd() / "../config/config.yaml").resolve()
with open(config_path, "r") as f:
    config = Munch.fromYAML(f)

load_dotenv((Path.cwd() / "../.env").resolve())
SUPABASE_PASSWORD = os.getenv("SUPABASE_PASSWORD")
GROQ_API_KEY      = os.getenv("GROQ_API_KEY")

if not SUPABASE_PASSWORD:
    raise ValueError("SUPABASE_PASSWORD not found in .env")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env")

DATABASE_URL = (
    f"postgresql://{config.supabase.user}:{SUPABASE_PASSWORD}@"
    f"{config.supabase.host}:{config.supabase.port}/{config.supabase.database}"
)
sql_dir    = (Path.cwd() / "sql").resolve()
assets_dir = (Path.cwd() / "../assets").resolve()

print("✅ Config and secrets loaded")

✅ Config and secrets loaded


In [3]:
print(f"Loading {config.embedding.model} …")
embed_model = BGEM3FlagModel(config.embedding.model, use_fp16=True)
print("✅ Embedding model loaded")

Loading BAAI/bge-m3 …


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ Embedding model loaded


---

## 1. Benchmark Metrics

Copied verbatim from Notebook 7 — **do not modify these**. Changing the scoring functions would make your results incomparable to the baseline.

In [4]:
RETRIEVAL_THRESHOLD = config.benchmarking.retrieval_threshold


def token_overlap(chunk: str, reference: str) -> float:
    """Fraction of reference tokens that appear in the chunk (recall-oriented)."""
    ref_tokens   = set(reference.lower().split())
    chunk_tokens = set(chunk.lower().split())
    if not ref_tokens:
        return 0.0
    return len(ref_tokens & chunk_tokens) / len(ref_tokens)


def hit_rate(contexts: list[str], reference: str) -> float:
    return float(
        any(token_overlap(chunk=ctx, reference=reference) >= RETRIEVAL_THRESHOLD
            for ctx in contexts)
    )


def mean_reciprocal_rank(contexts: list[str], reference: str) -> float:
    for rank, ctx in enumerate(contexts, start=1):
        if token_overlap(chunk=ctx, reference=reference) >= RETRIEVAL_THRESHOLD:
            return 1.0 / rank
    return 0.0


def mean_token_overlap(contexts: list[str], reference: str) -> float:
    if not contexts:
        return 0.0
    return sum(token_overlap(chunk=ctx, reference=reference) for ctx in contexts) / len(contexts)


def max_token_overlap(contexts: list[str], reference: str) -> float:
    if not contexts:
        return 0.0
    return max(token_overlap(chunk=ctx, reference=reference) for ctx in contexts)


def rouge_l(prediction: str, reference: str) -> float:
    scorer = rouge_scorer.RougeScorer(rouge_types=["rougeL"], use_stemmer=True)
    return scorer.score(target=reference, prediction=prediction)["rougeL"].fmeasure


def compute_bert_scores(predictions: list[str], references: list[str]) -> list[float]:
    _, _, f1 = bert_score(cands=predictions, refs=references, lang="en", verbose=False)
    return f1.tolist()


print(f"✅ Metric functions ready  (retrieval threshold = {RETRIEVAL_THRESHOLD})")

✅ Metric functions ready  (retrieval threshold = 0.15)


In [5]:
METRIC_COLS = ["hit_rate", "mrr", "mean_token_overlap", "max_token_overlap", "rouge_l", "bert_score"]


def evaluate_outputs(outputs: list[dict]) -> pd.DataFrame:
    """
    Score a list of RAG outputs with all six offline metrics.
    Identical to Notebook 7 — do not modify.

    Args:
        outputs : List of {question, contexts, answer, reference} dicts.

    Returns:
        DataFrame with one row per sample and one column per metric.
    """
    total = len(outputs)
    print(f"📊 Evaluating {total} samples …")

    predictions = [o["answer"]    for o in outputs]
    references  = [o["reference"] for o in outputs]

    print("🔍 Computing BERTScore …")
    bert_scores = compute_bert_scores(predictions=predictions, references=references)
    print("✅ BERTScore done")

    rows = []
    for i, o in enumerate(outputs):
        rows.append({
            "question":           o["question"],
            "reference":          o["reference"],
            "answer":             o["answer"],
            "hit_rate":           hit_rate(contexts=o["contexts"], reference=o["reference"]),
            "mrr":                mean_reciprocal_rank(contexts=o["contexts"], reference=o["reference"]),
            "mean_token_overlap": mean_token_overlap(contexts=o["contexts"], reference=o["reference"]),
            "max_token_overlap":  max_token_overlap(contexts=o["contexts"], reference=o["reference"]),
            "rouge_l":            rouge_l(prediction=o["answer"], reference=o["reference"]),
            "bert_score":         bert_scores[i],
        })

    print(f"✅ Evaluation complete ({total} samples, {len(METRIC_COLS)} metrics)")
    return pd.DataFrame(rows)


print("✅ evaluate_outputs() ready")

✅ evaluate_outputs() ready


---

## 2. Test Set

Same 100-question set used in Notebook 7. Adjust `SAMPLE_SIZE` if you want a quicker iteration loop while experimenting, but run the full set for your final score.

In [6]:
def load_test_set(filepath: Path) -> list[dict]:
    with open(filepath, "r", encoding="utf-8") as f:
        return list(csv.DictReader(f))


full_test_set = load_test_set(assets_dir / "benchmark_datasets/alice_test_set.csv")

# ── Adjust these to control speed vs coverage ─────────────────────────────────
SAMPLE_SIZE  = config.benchmarking.sample_size   # change to len(full_test_set) for full run
RANDOM_SEED  = config.benchmarking.random_seed
# ──────────────────────────────────────────────────────────────────────────────

random.seed(RANDOM_SEED)
test_sample = random.sample(full_test_set, k=min(SAMPLE_SIZE, len(full_test_set)))

print(f"Total available : {len(full_test_set)}")
print(f"Using           : {len(test_sample)} questions (seed={RANDOM_SEED})")

Total available : 100
Using           : 10 questions (seed=42)


---

## 3. Baseline Scores (Notebook 7)

These are the scores to beat. Enter them here so the final comparison cell can show your improvement.

> Update the values below with your actual Notebook 7 results.

In [7]:
# ── Paste your Notebook 7 mean scores here ────────────────────────────────────
BASELINE = {
    "hit_rate":           0.90,
    "mrr":                0.833,
    "mean_token_overlap": 0.284,
    "max_token_overlap":  0.470,
    "rouge_l":            0.203,
    "bert_score":         0.873,
    "latency_s":          None,   # fill in if you timed Notebook 7
}
# ──────────────────────────────────────────────────────────────────────────────

print("Baseline (Notebook 7):")
for k, v in BASELINE.items():
    print(f"  {k:<22} {v}")

Baseline (Notebook 7):
  hit_rate               0.9
  mrr                    0.833
  mean_token_overlap     0.284
  max_token_overlap      0.47
  rouge_l                0.203
  bert_score             0.873
  latency_s              None


---

## 4. Your RAG Pipeline

**This is the only section you need to edit.**

Implement `run_rag(question)` so that it returns a dict with keys `question`, `contexts` (list of strings, in rank order), and `answer` (string). How you get there is entirely up to you.

The helper `get_connection()` is provided since every approach will need database access. Everything else — retrieval strategy, fusion, re-ranking, prompt, LLM — is your design decision.

> 💡 **Tip**: change one thing at a time and re-run Section 5 after each change so you know what actually moved the needle.

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
#  YOUR CODE GOES HERE
#
#  Implement a run_rag(question: str) -> dict function.
#  The returned dict must have:
#    - "question"  : str
#    - "contexts"  : list[str]   (retrieved chunks, in rank order)
#    - "answer"    : str         (the LLM-generated answer)
#
#  Everything else is up to you.
# ══════════════════════════════════════════════════════════════════════════════


def get_connection() -> psycopg2.extensions.connection:
    return psycopg2.connect(DATABASE_URL)


# TODO: build your pipeline here




def run_rag(question: str) -> dict:
    """
    Run your RAG pipeline for a single question.

    Returns:
        Dict with keys: question, contexts (list[str]), answer (str).
    """
    raise NotImplementedError("Implement your pipeline here.")


print("✅ run_rag() defined — make sure it no longer raises NotImplementedError before running Section 5")

✅ run_rag() defined — make sure it no longer raises NotImplementedError before running Section 5


---

## 5. Run & Score

Once your pipeline is implemented, run these two cells. Wall-clock time is measured automatically.

In [9]:
def collect_outputs(test_samples: list[dict]) -> tuple[list[dict], float]:
    """
    Run run_rag() on every question, attach the reference answer, and
    return (outputs, total_wall_time_seconds).
    """
    outputs   = []
    t_start   = time.perf_counter()

    for i, row in enumerate(test_samples, start=1):
        print(f"  [{i}/{len(test_samples)}] {row['question'][:70]}…")
        result = run_rag(question=row["question"])
        outputs.append({
            "question":  row["question"],
            "contexts":  result["contexts"],
            "answer":    result["answer"],
            "reference": row["answer"],
        })

    elapsed = time.perf_counter() - t_start
    return outputs, elapsed


print("Running your pipeline …\n")
outputs, total_time = collect_outputs(test_sample)
latency_per_q = total_time / len(outputs)

print(f"\n✅ Done — {len(outputs)} questions in {total_time:.1f}s "
      f"({latency_per_q:.2f}s / question)")

Running your pipeline …

  [1/10] What four specific things beginning with an 'M' did the Dormouse claim…


NotImplementedError: Implement your pipeline here! Return {'question': ..., 'contexts': [...], 'answer': ...}

In [ ]:
results_df = evaluate_outputs(outputs)

summary = results_df[METRIC_COLS].mean().round(3)
print("\nMean scores:\n")
print(summary.to_string())

---

## 6. Comparison vs Baseline

In [ ]:
# Build comparison table
yours = summary.to_dict()
yours["latency_s"] = round(latency_per_q, 3)

all_keys = METRIC_COLS + ["latency_s"]
comparison = pd.DataFrame(
    {
        "baseline": [BASELINE.get(k) for k in all_keys],
        "yours":    [yours.get(k)     for k in all_keys],
    },
    index=all_keys,
)

def delta_str(row):
    if row["baseline"] is None or row["yours"] is None:
        return "n/a"
    d = row["yours"] - row["baseline"]
    # For latency, lower is better — flip the sign on the emoji
    if row.name == "latency_s":
        return f"{d:+.3f}  {'🟢' if d <= 0 else '🔴'}"
    return f"{d:+.3f}  {'🟢' if d >= 0 else '🔴'}"

comparison["delta"] = comparison.apply(delta_str, axis=1)
print(comparison.to_string())

In [ ]:
matplotlib.rcParams["font.family"] = "monospace"

plot_metrics = METRIC_COLS  # latency excluded from radar
baseline_vals = [BASELINE.get(m, 0) or 0 for m in plot_metrics]
yours_vals    = [yours.get(m, 0)     or 0 for m in plot_metrics]

x = range(len(plot_metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle("Your Pipeline vs Baseline — Mean Scores", fontsize=13)

bars_b = ax.bar([i - width / 2 for i in x], baseline_vals, width, label="Baseline (NB 7)",
                color="#6b6560", zorder=3)
bars_y = ax.bar([i + width / 2 for i in x], yours_vals,    width, label="Yours",
                color="#5b8dd9", zorder=3)

ax.set_xticks(list(x))
ax.set_xticklabels([m.replace("_", "\n") for m in plot_metrics], fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score (0–1)")
ax.yaxis.grid(visible=True, linestyle="--", alpha=0.4, zorder=0)
ax.set_axisbelow(True)
ax.legend()

for bar in list(bars_b) + list(bars_y):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{bar.get_height():.2f}",
        ha="center", va="bottom", fontsize=8,
    )

plt.tight_layout()
plt.show()

print(f"\n⏱  Latency — baseline: {BASELINE['latency_s'] or 'n/a'}s  |  yours: {latency_per_q:.2f}s per question")

---

## 7. Save Results

In [ ]:
output_dir = (Path.cwd() / "../assets/benchmark_results").resolve()
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(output_dir / "hackathon_results.csv", index=False)
print(f"✅ Results saved to {output_dir / 'hackathon_results.csv'}")
print(f"   Columns: {list(results_df.columns)}")